In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if root.count('/') <= 5:
        print(root, '->', len(files), 'files')

/kaggle/input -> 0 files
/kaggle/input/datasets -> 0 files
/kaggle/input/datasets/dartweichen -> 0 files
/kaggle/input/datasets/dartweichen/student-life -> 0 files


In [3]:
import os

BASE = '/kaggle/input/datasets/dartweichen/student-life'

for root, dirs, files in os.walk(BASE):
    depth = root.replace(BASE, '').count('/')
    if depth <= 3:
        print(root.replace(BASE, '.'), '->', len(files), 'files')

. -> 0 files
./dataset -> 0 files
./dataset/EMA -> 1 files
./dataset/EMA/response -> 0 files
./dataset/app_usage -> 49 files
./dataset/sensing -> 0 files
./dataset/sensing/conversation -> 49 files
./dataset/sensing/phonelock -> 49 files
./dataset/sensing/phonecharge -> 49 files
./dataset/sensing/dark -> 49 files
./dataset/sensing/audio -> 49 files
./dataset/sensing/gps -> 49 files
./dataset/sensing/wifi_location -> 49 files
./dataset/sensing/bluetooth -> 49 files
./dataset/sensing/wifi -> 49 files
./dataset/sensing/activity -> 49 files
./dataset/calendar -> 28 files
./dataset/call_log -> 49 files
./dataset/education -> 5 files
./dataset/survey -> 8 files
./dataset/dinning -> 31 files
./dataset/sms -> 49 files


In [5]:
import os, json

BASE = '/kaggle/input/datasets/dartweichen/student-life/dataset'

# 1. Which EMA surveys exist?
print("EMA response folders:")
print(sorted(os.listdir(f'{BASE}/EMA/response')))

# 2. How many students answered the Stress survey?
stress_dir = f'{BASE}/EMA/response/Stress'
stress_files = sorted(os.listdir(stress_dir))
print(f"\nStress files: {len(stress_files)}")
print(stress_files[:5])

# 3. What did the Stress question actually ask?
defn = json.load(open(f'{BASE}/EMA/EMA_definition.json'))
print("\nSTRESS QUESTION DEFINITION:")
if isinstance(defn, list):
    for item in defn:
        if 'stress' in str(item.get('name', '')).lower():
            print(json.dumps(item, indent=2))
else:
    for k, v in defn.items():
        if 'stress' in k.lower():
            print(k, json.dumps(v, indent=2))

# 4. What does one student's answers look like?
resp = json.load(open(f'{stress_dir}/{stress_files[0]}'))
print(f"\nSample from {stress_files[0]}: {len(resp)} responses")
print(json.dumps(resp[:3], indent=2))

EMA response folders:
['Activity', 'Administration response', 'Behavior', 'Boston Bombing', 'Cancelled Classes', 'Class', 'Class 2', 'Comment', 'Dartmouth now', 'Dimensions', 'Dimensions protestors', 'Dining Halls', 'Events', 'Exercise', 'Green Key 1', 'Green Key 2', 'Lab', 'Mood', 'Mood 1', 'Mood 2', 'PAM', 'QR_Code', 'Sleep', 'Social', 'Stress', 'Study Spaces']

Stress files: 49
['Stress_u00.json', 'Stress_u01.json', 'Stress_u02.json', 'Stress_u03.json', 'Stress_u04.json']

STRESS QUESTION DEFINITION:
{
  "name": "Stress",
  "questions": [
    {
      "options": "[1]A little stressed, [2]Definitely stressed, [3]Stressed out, [4]Feeling good, [5]Feeling great, ",
      "question_id": "level",
      "question_text": "Right now, I am..."
    },
    {
      "options": "",
      "question_id": "location",
      "question_text": ""
    }
  ]
}

Sample from Stress_u00.json: 79 responses
[
  {
    "null": "43.75908069,-72.32885314",
    "resp_time": 1364114401
  },
  {
    "null": "43.759080

In [6]:
import os, json
from collections import Counter

BASE = '/kaggle/input/datasets/dartweichen/student-life/dataset'
stress_dir = f'{BASE}/EMA/response/Stress'

total, with_level = 0, 0
levels = Counter()
per_user = {}

for fn in sorted(os.listdir(stress_dir)):
    if not fn.endswith('.json'):
        continue
    uid = fn.replace('Stress_', '').replace('.json', '')
    try:
        data = json.load(open(f'{stress_dir}/{fn}'))
    except Exception as e:
        print('could not read', fn, e)
        continue

    n_ok = 0
    for r in data:
        total += 1
        lv = r.get('level')
        if lv is not None:
            with_level += 1
            levels[int(lv)] += 1
            n_ok += 1
    per_user[uid] = n_ok

print(f"total responses       : {total}")
print(f"with a 'level' answer : {with_level}  ({with_level/total:.1%})")
print(f"\nlevel distribution    : {dict(sorted(levels.items()))}")

counts = sorted(per_user.values())
print(f"\nusers                 : {len(per_user)}")
print(f"usable per user       : min {counts[0]}, "
      f"median {counts[len(counts)//2]}, max {counts[-1]}")
print(f"users with >= 20      : {sum(1 for c in counts if c >= 20)}")

total responses       : 2408
with a 'level' answer : 2167  (90.0%)

level distribution    : {1: 974, 2: 347, 3: 266, 4: 457, 5: 123}

users                 : 49
usable per user       : min 0, median 37, max 269
users with >= 20      : 36


In [7]:
import os, json

BASE = '/kaggle/input/datasets/dartweichen/student-life/dataset'
stress_dir = f'{BASE}/EMA/response/Stress'

LABEL_MAP = {2: 1, 3: 1, 4: 0, 5: 0}   # Option C: level 1 dropped

rows = []
for fn in sorted(os.listdir(stress_dir)):
    if not fn.endswith('.json'):
        continue
    uid = fn.replace('Stress_', '').replace('.json', '')
    for r in json.load(open(f'{stress_dir}/{fn}')):
        lv = r.get('level')
        if lv is None:
            continue
        lv = int(lv)
        if lv in LABEL_MAP:
            rows.append((uid, int(r['resp_time']), lv, LABEL_MAP[lv]))

import pandas as pd
labels = pd.DataFrame(rows, columns=['uid', 'resp_time', 'level', 'y'])
print(f"samples: {len(labels)}   positive rate: {labels.y.mean():.1%}\n")

g = labels.groupby('uid').agg(n=('y', 'size'), pos_rate=('y', 'mean'))
both = g[(g.pos_rate > 0) & (g.pos_rate < 1)]

print(f"users total                    : {len(g)}")
print(f"users with >= 20 samples       : {(g.n >= 20).sum()}")
print(f"users with BOTH classes        : {len(both)}")
print(f"users with >= 20 AND both      : {((g.n >= 20) & (g.pos_rate > 0) & (g.pos_rate < 1)).sum()}")
print(f"\nper-user positive rate: min {g.pos_rate.min():.2f}, "
      f"max {g.pos_rate.max():.2f}, std {g.pos_rate.std():.3f}")

labels.to_csv('/kaggle/working/labels.csv', index=False)
print("\nsaved -> /kaggle/working/labels.csv")

samples: 1193   positive rate: 51.4%

users total                    : 47
users with >= 20 samples       : 23
users with BOTH classes        : 40
users with >= 20 AND both      : 21

per-user positive rate: min 0.00, max 1.00, std 0.326

saved -> /kaggle/working/labels.csv


In [8]:
import os, numpy as np, pandas as pd

BASE = '/kaggle/input/datasets/dartweichen/student-life/dataset'
labels = pd.read_csv('/kaggle/working/labels.csv')
WINDOW = 24 * 3600

def load_intervals(path):
    if not os.path.exists(path):
        return np.empty((0, 2), dtype=np.int64)
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    s = [c for c in df.columns if 'start' in c][0]
    e = [c for c in df.columns if 'end' in c][0]
    a = df[[s, e]].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(np.int64)
    return a[a[:, 1] > a[:, 0]]

def load_points(path):
    if not os.path.exists(path):
        return np.empty(0, np.int64), np.empty(0, np.int64)
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    t = [c for c in df.columns if 'time' in c][0]
    v = [c for c in df.columns if 'inference' in c][0]
    df = df[[t, v]].apply(pd.to_numeric, errors='coerce').dropna()
    ts, vs = df[t].to_numpy(np.int64), df[v].to_numpy(np.int64)
    o = np.argsort(ts)
    return ts[o], vs[o]

def overlap(iv, w0, w1):
    if len(iv) == 0:
        return np.zeros(0)
    return np.clip(np.minimum(iv[:, 1], w1) - np.maximum(iv[:, 0], w0), 0, None)

rows = []
uids = sorted(labels.uid.unique())

for i, uid in enumerate(uids, 1):
    dark = load_intervals(f'{BASE}/sensing/dark/dark_{uid}.csv')
    lock = load_intervals(f'{BASE}/sensing/phonelock/phonelock_{uid}.csv')
    conv = load_intervals(f'{BASE}/sensing/conversation/conversation_{uid}.csv')
    a_ts, a_v = load_points(f'{BASE}/sensing/activity/activity_{uid}.csv')
    u_ts, u_v = load_points(f'{BASE}/sensing/audio/audio_{uid}.csv')

    sub = labels[labels.uid == uid]
    for _, r in sub.iterrows():
        w1 = int(r.resp_time)
        w0 = w1 - WINDOW

        d = overlap(dark, w0, w1)
        l = overlap(lock, w0, w1)
        c = overlap(conv, w0, w1)

        i0, i1 = np.searchsorted(a_ts, [w0, w1])
        act = a_v[i0:i1]; act = act[act != 3]

        j0, j1 = np.searchsorted(u_ts, [w0, w1])
        aud = u_v[j0:j1]; aud = aud[aud != 3]

        rows.append({
            'uid': uid, 'resp_time': w1, 'y': int(r.y), 'level': int(r.level),
            'sleep_hours':    d.max() / 3600 if len(d) else 0.0,
            'dark_hours':     d.sum() / 3600,
            'idle_hours':     l.sum() / 3600,
            'idle_episodes':  int(((lock[:, 1] >= w0) & (lock[:, 1] < w1)).sum()) if len(lock) else 0,
            'conv_count':     int((c > 0).sum()),
            'conv_hours':     c.sum() / 3600,
            'activity_n':     len(act),
            'activity_level': float(np.isin(act, [1, 2]).mean()) if len(act) else np.nan,
            'audio_n':        len(aud),
            'voice_ratio':    float((aud == 1).mean()) if len(aud) else np.nan,
        })

    print(f"[{i:>2}/{len(uids)}] {uid}: {len(sub)} windows")

feat = pd.DataFrame(rows)
feat.to_csv('/kaggle/working/features.csv', index=False)

print("\nshape:", feat.shape)
print("\nmissing values:\n", feat.isna().sum()[lambda s: s > 0])
print("\n", feat.describe().T[['mean', 'std', 'min', 'max']])

[ 1/47] u00: 44 windows
[ 2/47] u01: 17 windows
[ 3/47] u02: 15 windows
[ 4/47] u03: 19 windows
[ 5/47] u04: 22 windows
[ 6/47] u05: 6 windows
[ 7/47] u07: 39 windows
[ 8/47] u08: 50 windows
[ 9/47] u09: 2 windows
[10/47] u10: 56 windows
[11/47] u12: 15 windows
[12/47] u14: 33 windows
[13/47] u15: 2 windows
[14/47] u16: 67 windows
[15/47] u17: 41 windows
[16/47] u18: 6 windows
[17/47] u19: 54 windows
[18/47] u20: 3 windows
[19/47] u22: 28 windows
[20/47] u23: 8 windows
[21/47] u24: 23 windows
[22/47] u25: 13 windows
[23/47] u27: 12 windows
[24/47] u30: 20 windows
[25/47] u31: 6 windows
[26/47] u32: 32 windows
[27/47] u33: 49 windows
[28/47] u34: 2 windows
[29/47] u35: 19 windows
[30/47] u36: 16 windows
[31/47] u41: 8 windows
[32/47] u42: 21 windows
[33/47] u43: 33 windows
[34/47] u44: 35 windows
[35/47] u45: 18 windows
[36/47] u46: 25 windows
[37/47] u47: 6 windows
[38/47] u49: 50 windows
[39/47] u50: 4 windows
[40/47] u51: 15 windows
[41/47] u52: 25 windows
[42/47] u53: 13 windows
[43

In [9]:
import os, numpy as np, pandas as pd

BASE = '/kaggle/input/datasets/dartweichen/student-life/dataset'
feat = pd.read_csv('/kaggle/working/features.csv')
WINDOW = 24 * 3600

def load_intervals(path):
    if not os.path.exists(path):
        return np.empty((0, 2), dtype=np.int64)
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    s = [c for c in df.columns if 'start' in c][0]
    e = [c for c in df.columns if 'end' in c][0]
    a = df[[s, e]].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(np.int64)
    return a[a[:, 1] > a[:, 0]]

def merge_intervals(iv):
    """Collapse overlapping/touching intervals so nothing is double counted."""
    if len(iv) == 0:
        return iv
    iv = iv[np.argsort(iv[:, 0])]
    out = [list(iv[0])]
    for s, e in iv[1:]:
        if s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return np.array(out, dtype=np.int64)

def overlap(iv, w0, w1):
    if len(iv) == 0:
        return np.zeros(0)
    return np.clip(np.minimum(iv[:, 1], w1) - np.maximum(iv[:, 0], w0), 0, None)

fixed = {}
for uid in sorted(feat.uid.unique()):
    dark = merge_intervals(load_intervals(f'{BASE}/sensing/dark/dark_{uid}.csv'))
    lock = merge_intervals(load_intervals(f'{BASE}/sensing/phonelock/phonelock_{uid}.csv'))
    conv = merge_intervals(load_intervals(f'{BASE}/sensing/conversation/conversation_{uid}.csv'))

    for idx, w1 in feat.loc[feat.uid == uid, 'resp_time'].items():
        w0 = w1 - WINDOW
        d, l, c = overlap(dark, w0, w1), overlap(lock, w0, w1), overlap(conv, w0, w1)
        fixed[idx] = {
            'sleep_hours':   d.max() / 3600 if len(d) else 0.0,
            'dark_hours':    d.sum() / 3600,
            'idle_hours':    l.sum() / 3600,
            'idle_episodes': int(((lock[:, 1] >= w0) & (lock[:, 1] < w1)).sum()) if len(lock) else 0,
            'conv_count':    int((c > 0).sum()),
            'conv_hours':    c.sum() / 3600,
        }

fix_df = pd.DataFrame.from_dict(fixed, orient='index')
before = feat[['dark_hours', 'idle_hours']].max()
feat.update(fix_df)

print("max hours BEFORE merge:\n", before, sep='')
print("\nmax hours AFTER merge:\n", feat[['sleep_hours','dark_hours','idle_hours','conv_hours']].max(), sep='')

# --- sensor coverage: how much data actually existed per window? ---
print("\nactivity_n percentiles:", 
      np.percentile(feat.activity_n.dropna(), [1, 5, 10, 25, 50]).round().tolist())
print("rows surviving activity_n threshold:")
for t in [0, 100, 500, 1000, 2000]:
    print(f"   >= {t:>5}: {(feat.activity_n >= t).sum():>5} of {len(feat)}")

feat.to_csv('/kaggle/working/features.csv', index=False)
print("\nsaved")

max hours BEFORE merge:
dark_hours    23.052778
idle_hours    27.718889
dtype: float64

max hours AFTER merge:
sleep_hours    17.758611
dark_hours     23.052778
idle_hours     23.973889
conv_hours     15.091667
dtype: float64

activity_n percentiles: [0.0, 2430.0, 4997.0, 7645.0, 8005.0]
rows surviving activity_n threshold:
   >=     0:  1193 of 1193
   >=   100:  1159 of 1193
   >=   500:  1158 of 1193
   >=  1000:  1155 of 1193
   >=  2000:  1141 of 1193

saved


In [10]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score)

feat = pd.read_csv('/kaggle/working/features.csv')

MIN_ACTIVITY = 2000
df = feat[feat.activity_n >= MIN_ACTIVITY].dropna(
        subset=['activity_level', 'voice_ratio']).copy()

FEATURES = ['sleep_hours', 'dark_hours', 'idle_hours', 'idle_episodes',
            'conv_count', 'conv_hours', 'activity_level', 'voice_ratio']

X, y, g = df[FEATURES].values, df.y.values, df.uid.values
print(f"samples {len(df)}   users {df.uid.nunique()}   positive {y.mean():.1%}")
print(f"MAJORITY BASELINE: {max(y.mean(), 1-y.mean()):.1%}\n")

def evaluate(model_fn, split, seeds=5, folds=5):
    out = []
    for s in range(seeds):
        cv = (StratifiedGroupKFold(folds, shuffle=True, random_state=s) if split == 'user'
              else StratifiedKFold(folds, shuffle=True, random_state=s))
        it = cv.split(X, y, g) if split == 'user' else cv.split(X, y)
        for tr, te in it:
            m = model_fn().fit(X[tr], y[tr])
            p = m.predict(X[te])
            pr = m.predict_proba(X[te])[:, 1]
            out.append([accuracy_score(y[te], p),
                        balanced_accuracy_score(y[te], p),
                        f1_score(y[te], p, zero_division=0),
                        roc_auc_score(y[te], pr)])
    a = np.array(out)
    return a.mean(0), a.std(0)

MODELS = {
    'Logistic Regression': lambda: make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1),
}

for split, note in [('user', 'USER-LEVEL SPLIT  (correct: no user in both train and test)'),
                    ('random', 'RANDOM SPLIT      (leaky: same user on both sides)')]:
    print("=" * 68)
    print(note)
    print("=" * 68)
    print(f"{'model':<22}{'acc':>14}{'bal-acc':>14}{'F1':>13}{'AUC':>13}")
    for name, fn in MODELS.items():
        mu, sd = evaluate(fn, split)
        print(f"{name:<22}" + "".join(
            f"{mu[i]:>9.3f}±{sd[i]:.3f}" for i in range(4)))
    print()

samples 1141   users 47   positive 51.6%
MAJORITY BASELINE: 51.6%

USER-LEVEL SPLIT  (correct: no user in both train and test)
model                            acc       bal-acc           F1          AUC
Logistic Regression       0.477±0.066    0.511±0.051    0.550±0.060    0.522±0.079
Random Forest             0.464±0.057    0.490±0.042    0.537±0.061    0.485±0.064

RANDOM SPLIT      (leaky: same user on both sides)
model                            acc       bal-acc           F1          AUC
Logistic Regression       0.575±0.027    0.573±0.027    0.615±0.035    0.602±0.036
Random Forest             0.642±0.022    0.640±0.022    0.668±0.028    0.690±0.026



In [11]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

feat = pd.read_csv('/kaggle/working/features.csv')
df = feat[feat.activity_n >= 2000].dropna(
        subset=['activity_level', 'voice_ratio']).copy()

FEATURES = ['sleep_hours', 'dark_hours', 'idle_hours', 'idle_episodes',
            'conv_count', 'conv_hours', 'activity_level', 'voice_ratio']

# ---- within-person z-scores: deviation from that user's own norm ----
def zs(s):
    sd = s.std()
    return (s - s.mean()) / (sd if sd and sd > 0 else 1.0)

Z = df.groupby('uid')[FEATURES].transform(zs).values
R = df[FEATURES].values
y, g = df.y.values, df.uid.values

MODELS = {
    'LogReg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'RF':     lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                             random_state=0, n_jobs=-1),
}

print("=" * 60)
print("GLOBAL MODEL, user-level split")
print("=" * 60)
print(f"{'features':<12}{'model':<10}{'bal-acc':>15}{'AUC':>15}")

for tag, Xm in [('raw', R), ('within-person', Z)]:
    for name, fn in MODELS.items():
        res = []
        for s in range(5):
            cv = StratifiedGroupKFold(5, shuffle=True, random_state=s)
            for tr, te in cv.split(Xm, y, g):
                m = fn().fit(Xm[tr], y[tr])
                res.append([balanced_accuracy_score(y[te], m.predict(Xm[te])),
                            roc_auc_score(y[te], m.predict_proba(Xm[te])[:, 1])])
        a = np.array(res)
        print(f"{tag:<12}{name:<10}" +
              "".join(f"{a.mean(0)[i]:>10.3f}±{a.std(0)[i]:.3f}" for i in range(2)))

# ---- fully personal models: train and test within one user only ----
print("\n" + "=" * 60)
print("PERSONAL MODEL, per-user CV (no cross-user transfer needed)")
print("=" * 60)

per_user = []
for uid, sub in df.groupby('uid'):
    if len(sub) < 20 or sub.y.nunique() < 2 or sub.y.sum() < 5 or (1 - sub.y).sum() < 5:
        continue
    Xu = sub.groupby('uid')[FEATURES].transform(zs).values
    yu = sub.y.values
    sc = []
    for s in range(5):
        cv = StratifiedKFold(3, shuffle=True, random_state=s)
        for tr, te in cv.split(Xu, yu):
            m = make_pipeline(StandardScaler(),
                              LogisticRegression(max_iter=2000)).fit(Xu[tr], yu[tr])
            if len(np.unique(yu[te])) > 1:
                sc.append(roc_auc_score(yu[te], m.predict_proba(Xu[te])[:, 1]))
    if sc:
        per_user.append((uid, len(sub), np.mean(sc)))

pu = pd.DataFrame(per_user, columns=['uid', 'n', 'auc'])
print(f"users evaluated: {len(pu)}")
print(f"mean personal AUC: {pu.auc.mean():.3f}  (std {pu.auc.std():.3f})")
print(f"users above 0.60 : {(pu.auc > 0.60).sum()} / {len(pu)}")
print("\nbest and worst:")
print(pu.sort_values('auc', ascending=False).head(5).to_string(index=False))
print(pu.sort_values('auc').head(5).to_string(index=False))

GLOBAL MODEL, user-level split
features    model             bal-acc            AUC
raw         LogReg         0.511±0.051     0.522±0.079
raw         RF             0.490±0.042     0.485±0.064
within-personLogReg         0.528±0.027     0.558±0.031
within-personRF             0.540±0.035     0.567±0.044

PERSONAL MODEL, per-user CV (no cross-user transfer needed)
users evaluated: 13
mean personal AUC: 0.614  (std 0.155)
users above 0.60 : 8 / 13

best and worst:
uid   n      auc
u43  33 0.962963
u10  56 0.733175
u44  33 0.677778
u49  50 0.655960
u59 119 0.649253
uid  n      auc
u52 23 0.293333
u19 54 0.425873
u00 38 0.542857
u46 25 0.578148
u58 31 0.580093


In [12]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from scipy import stats

feat = pd.read_csv('/kaggle/working/features.csv')
df = feat[feat.activity_n >= 2000].dropna(
        subset=['activity_level', 'voice_ratio']).copy()

FEATURES = ['sleep_hours', 'dark_hours', 'idle_hours', 'idle_episodes',
            'conv_count', 'conv_hours', 'activity_level', 'voice_ratio']

def zs(s):
    sd = s.std()
    return (s - s.mean()) / (sd if sd and sd > 0 else 1.0)

MIN_N, MIN_PER_CLASS, N_REPEATS = 15, 4, 10

rows = []
for uid, sub in df.groupby('uid'):
    n = len(sub); pos = int(sub.y.sum()); neg = n - pos
    if n < MIN_N or pos < MIN_PER_CLASS or neg < MIN_PER_CLASS:
        continue
    Xu = sub[FEATURES].apply(zs).values
    yu = sub.y.values

    aucs, bals = [], []
    for r in range(N_REPEATS):
        oof = np.full(n, np.nan)
        for tr, te in StratifiedKFold(3, shuffle=True, random_state=r).split(Xu, yu):
            m = make_pipeline(StandardScaler(),
                              LogisticRegression(max_iter=2000)).fit(Xu[tr], yu[tr])
            oof[te] = m.predict_proba(Xu[te])[:, 1]
        aucs.append(roc_auc_score(yu, oof))
        bals.append(balanced_accuracy_score(yu, (oof >= 0.5).astype(int)))

    rows.append({'uid': uid, 'n': n, 'pos_rate': round(pos / n, 2),
                 'auc': np.mean(aucs), 'auc_sd': np.std(aucs),
                 'bal_acc': np.mean(bals)})

pu = pd.DataFrame(rows).sort_values('auc', ascending=False)

print(f"users evaluated : {len(pu)}   (was 13)")
print(f"mean AUC        : {pu.auc.mean():.3f}  (sd {pu.auc.std():.3f})")
print(f"median AUC      : {pu.auc.median():.3f}")
print(f"mean bal-acc    : {pu.bal_acc.mean():.3f}")
print(f"users > 0.60    : {(pu.auc > 0.60).sum()} / {len(pu)}")
print(f"users > 0.50    : {(pu.auc > 0.50).sum()} / {len(pu)}")

t, p = stats.ttest_1samp(pu.auc, 0.5)
try:
    w, pw = stats.wilcoxon(pu.auc - 0.5)
except ValueError:
    pw = np.nan
print(f"\nH0: mean AUC = 0.50")
print(f"  t-test    t={t:.2f}  p={p:.4f}")
print(f"  Wilcoxon           p={pw:.4f}")

# leave-one-user-out: is the result driven by a single person?
loo = [np.delete(pu.auc.values, i).mean() for i in range(len(pu))]
print(f"\nleave-one-user-out mean AUC: "
      f"min {min(loo):.3f}, max {max(loo):.3f}")
print(f"  (if min stays well above 0.50, no single user drives the result)")

print(f"\ncorrelation of AUC with sample size: "
      f"r={pu[['n','auc']].corr().iloc[0,1]:.2f}")

print("\n", pu.to_string(index=False))
pu.to_csv('/kaggle/working/personal_results.csv', index=False)

users evaluated : 20   (was 13)
mean AUC        : 0.599  (sd 0.192)
median AUC      : 0.584
mean bal-acc    : 0.564
users > 0.60    : 9 / 20
users > 0.50    : 14 / 20

H0: mean AUC = 0.50
  t-test    t=2.32  p=0.0319
  Wilcoxon           p=0.0583

leave-one-user-out mean AUC: min 0.580, max 0.613
  (if min stays well above 0.50, no single user drives the result)

correlation of AUC with sample size: r=0.06

 uid   n  pos_rate      auc   auc_sd  bal_acc
u43  33      0.82 0.962346 0.001852 0.906481
u03  19      0.32 0.910256 0.040542 0.787821
u35  19      0.26 0.904286 0.039409 0.761429
u01  16      0.44 0.828571 0.033895 0.796825
u10  56      0.21 0.717235 0.061779 0.595833
u44  33      0.30 0.670870 0.065508 0.618478
u59 119      0.22 0.638462 0.019814 0.567639
u49  50      0.30 0.634095 0.042090 0.526190
u33  44      0.55 0.620417 0.033809 0.627500
u46  25      0.68 0.592647 0.082550 0.493382
u16  67      0.84 0.575487 0.041345 0.504058
u57  30      0.73 0.573295 0.070850 0.518750
u58

In [13]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

feat = pd.read_csv('/kaggle/working/features.csv')
df = feat[feat.activity_n >= 2000].dropna(subset=['activity_level','voice_ratio']).copy()

FEATURES = ['sleep_hours','dark_hours','idle_hours','idle_episodes',
            'conv_count','conv_hours','activity_level','voice_ratio']

MIN_N, MIN_PER_CLASS = 15, 4
clients = [u for u, s in df.groupby('uid')
           if len(s) >= MIN_N and s.y.sum() >= MIN_PER_CLASS
           and (1-s.y).sum() >= MIN_PER_CLASS]
print(f"clients: {len(clients)}\n")

ROUNDS, LOCAL_EPOCHS, LR, L2, SEEDS = 30, 5, 0.5, 0.01, 5

def sig(z): return 1/(1+np.exp(-np.clip(z,-30,30)))

def train(X, y, w, b, epochs):
    n = len(y)
    for _ in range(epochs):
        p = sig(X @ w + b)
        w -= LR * (X.T @ (p-y)/n + L2*w)
        b -= LR * (p-y).mean()
    return w, b

results = {k: [] for k in ['Local','FedAvg','PersonalizedFL','Centralized']}
curve = np.zeros((SEEDS, ROUNDS))

for seed in range(SEEDS):
    # build one train/test split per client (same splits for all 4 conditions)
    parts = {}
    for uid in clients:
        s = df[df.uid == uid]
        Xr, yr = s[FEATURES].values, s.y.values
        tr, te = next(iter(StratifiedKFold(3, shuffle=True,
                                           random_state=seed).split(Xr, yr)))
        mu, sd = Xr[tr].mean(0), Xr[tr].std(0)          # fit on TRAIN only
        sd[sd == 0] = 1.0
        parts[uid] = ((Xr[tr]-mu)/sd, yr[tr], (Xr[te]-mu)/sd, yr[te])

    d = len(FEATURES)

    # ---- 1. LOCAL: each client alone ----
    for uid in clients:
        Xt, yt, Xe, ye = parts[uid]
        w, b = train(Xt, yt, np.zeros(d), 0.0, ROUNDS*LOCAL_EPOCHS)
        if len(np.unique(ye)) > 1:
            results['Local'].append(roc_auc_score(ye, sig(Xe@w+b)))

    # ---- 2. FEDAVG: weighted aggregation over rounds ----
    gw, gb = np.zeros(d), 0.0
    for r in range(ROUNDS):
        ws, bs, ns = [], [], []
        for uid in clients:
            Xt, yt, _, _ = parts[uid]
            w, b = train(Xt, yt, gw.copy(), gb, LOCAL_EPOCHS)
            ws.append(w); bs.append(b); ns.append(len(yt))
        wt = np.array(ns) / sum(ns)
        gw = (np.array(ws) * wt[:,None]).sum(0)
        gb = float((np.array(bs) * wt).sum())

        a = [roc_auc_score(ye, sig(Xe@gw+gb))
             for uid in clients
             for _,_,Xe,ye in [parts[uid]] if len(np.unique(ye)) > 1]
        curve[seed, r] = np.mean(a)

    for uid in clients:
        _,_,Xe,ye = parts[uid]
        if len(np.unique(ye)) > 1:
            results['FedAvg'].append(roc_auc_score(ye, sig(Xe@gw+gb)))

    # ---- 3. PERSONALIZED FL: global model, then fine-tune locally ----
    for uid in clients:
        Xt, yt, Xe, ye = parts[uid]
        w, b = train(Xt, yt, gw.copy(), gb, LOCAL_EPOCHS*5)
        if len(np.unique(ye)) > 1:
            results['PersonalizedFL'].append(roc_auc_score(ye, sig(Xe@w+b)))

    # ---- 4. CENTRALIZED: pooled training data ----
    Xa = np.vstack([parts[u][0] for u in clients])
    ya = np.concatenate([parts[u][1] for u in clients])
    w, b = train(Xa, ya, np.zeros(d), 0.0, ROUNDS*LOCAL_EPOCHS)
    for uid in clients:
        _,_,Xe,ye = parts[uid]
        if len(np.unique(ye)) > 1:
            results['Centralized'].append(roc_auc_score(ye, sig(Xe@w+b)))

print(f"{'condition':<18}{'AUC':>16}{'n':>7}")
print("-"*41)
for k in ['Centralized','FedAvg','Local','PersonalizedFL']:
    a = np.array(results[k])
    print(f"{k:<18}{a.mean():>9.3f}±{a.std():.3f}{len(a):>7}")

from scipy import stats
loc, pfl = np.array(results['Local']), np.array(results['PersonalizedFL'])
fed = np.array(results['FedAvg'])
print(f"\nPFL vs Local : Δ={pfl.mean()-loc.mean():+.3f}  "
      f"p={stats.wilcoxon(pfl,loc).pvalue:.4f}")
print(f"PFL vs FedAvg: Δ={pfl.mean()-fed.mean():+.3f}  "
      f"p={stats.wilcoxon(pfl,fed).pvalue:.4f}")

m = curve.mean(0)
print(f"\nFedAvg convergence (mean AUC by round):")
for r in [0,4,9,19,29]:
    print(f"  round {r+1:>2}: {m[r]:.3f}")
np.save('/kaggle/working/fedavg_curve.npy', curve)
pd.DataFrame(results).to_csv('/kaggle/working/fl_comparison.csv', index=False)


clients: 20

condition                      AUC      n
-----------------------------------------
Centralized           0.615±0.223    100
FedAvg                0.615±0.219    100
Local                 0.628±0.255    100
PersonalizedFL        0.616±0.254    100

PFL vs Local : Δ=-0.012  p=0.1330
PFL vs FedAvg: Δ=+0.002  p=0.5899

FedAvg convergence (mean AUC by round):
  round  1: 0.608
  round  5: 0.618
  round 10: 0.620
  round 20: 0.618
  round 30: 0.615


In [14]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold

feat = pd.read_csv('/kaggle/working/features.csv')
df = feat[feat.activity_n >= 2000].dropna(subset=['activity_level','voice_ratio']).copy()
FEATURES = ['sleep_hours','dark_hours','idle_hours','idle_episodes',
            'conv_count','conv_hours','activity_level','voice_ratio']

clients = [u for u, s in df.groupby('uid')
           if len(s) >= 15 and s.y.sum() >= 4 and (1-s.y).sum() >= 4]
df = df[df.uid.isin(clients)].reset_index(drop=True)
K = len(clients)
print(f"clients: {K}   samples: {len(df)}\n")

# ONE global standardisation, identical for both schemes,
# so client partitioning is the only thing that differs.
Xg = ((df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()).values
yg = df.y.values

ROUNDS, EPOCHS, LR, L2, SEEDS = 30, 5, 0.5, 0.01, 5
def sig(z): return 1/(1+np.exp(-np.clip(z,-30,30)))
def train(X, y, w, b, ep):
    n = len(y)
    for _ in range(ep):
        p = sig(X@w+b); w -= LR*(X.T@(p-y)/n + L2*w); b -= LR*(p-y).mean()
    return w, b

def run(shards, seed):
    parts = {}
    for k, idx in enumerate(shards):
        Xs, ys = Xg[idx], yg[idx]
        if len(np.unique(ys)) < 2: continue
        tr, te = next(iter(StratifiedKFold(3, shuffle=True,
                                           random_state=seed).split(Xs, ys)))
        parts[k] = (Xs[tr], ys[tr], Xs[te], ys[te])
    d = len(FEATURES)

    loc = [roc_auc_score(ye, sig(Xe@w+b))
           for k,(Xt,yt,Xe,ye) in parts.items()
           for w,b in [train(Xt,yt,np.zeros(d),0.0,ROUNDS*EPOCHS)]
           if len(np.unique(ye))>1]

    gw, gb = np.zeros(d), 0.0
    for _ in range(ROUNDS):
        ws, bs, ns = [], [], []
        for Xt,yt,_,_ in parts.values():
            w,b = train(Xt,yt,gw.copy(),gb,EPOCHS)
            ws.append(w); bs.append(b); ns.append(len(yt))
        wt = np.array(ns)/sum(ns)
        gw = (np.array(ws)*wt[:,None]).sum(0); gb = float((np.array(bs)*wt).sum())

    fed = [roc_auc_score(ye, sig(Xe@gw+gb))
           for _,_,Xe,ye in parts.values() if len(np.unique(ye))>1]
    het = np.std([yt.mean() for _,yt,_,_ in parts.values()])
    return np.mean(loc), np.mean(fed), het

from sklearn.metrics import roc_auc_score
out = {}
for scheme in ['user-level', 'k-fold']:
    L, F, H = [], [], []
    for seed in range(SEEDS):
        if scheme == 'user-level':
            shards = [df.index[df.uid == u].values for u in clients]
        else:
            perm = np.random.RandomState(seed).permutation(len(df))
            shards = np.array_split(perm, K)
        l, f, h = run(shards, seed)
        L.append(l); F.append(f); H.append(h)
    out[scheme] = (np.mean(L), np.mean(F), np.mean(H))

print(f"{'partitioning':<16}{'Local':>9}{'FedAvg':>9}{'Δ':>9}{'label het σ':>13}")
print("-"*56)
for s,(l,f,h) in out.items():
    print(f"{s:<16}{l:>9.3f}{f:>9.3f}{f-l:>+9.3f}{h:>13.3f}")

clients: 20   samples: 733

partitioning        Local   FedAvg        Δ  label het σ
--------------------------------------------------------
user-level          0.611    0.625   +0.014        0.231
k-fold              0.538    0.602   +0.064        0.076
